# <font color='blue'>Data Science Academy</font>
# <font color='blue'>Deep Learning Para Aplicações de IA com PyTorch e Lightning</font>

## <font color='blue'>Mini-Projeto 4</font>
## <font color='blue'>Segmentação Automática de Imagens com SAM (Segment Anything Model)</font>

https://segment-anything.com/

![DSA](images/MP4.png)

## Instalando e Carregando os Pacotes

In [ ]:
# Versão da Linguagem Python
from platform import python_version
print('Versão da Linguagem Python Usada Neste Jupyter Notebook:', python_version())

In [ ]:
# Para atualizar um pacote, execute o comando abaixo no terminal ou prompt de comando:
# pip install -U nome_pacote

# Para instalar a versão exata de um pacote, execute o comando abaixo no terminal ou prompt de comando:
# !pip install nome_pacote==versão_desejada

# Depois de instalar ou atualizar o pacote, reinicie o jupyter notebook.

# Instala o pacote watermark. 
# Esse pacote é usado para gravar as versões de outros pacotes usados neste jupyter notebook.
!pip install -q -U watermark

In [ ]:
import os
HOME = os.getcwd()
print("HOME:", HOME)

Instale Segment Anything Model (SAM) e outras dependências.

https://github.com/facebookresearch

In [ ]:
%cd {HOME}

import sys
!{sys.executable} -m pip install 'git+https://github.com/facebookresearch/segment-anything.git'

In [ ]:
# https://pypi.org/project/opencv-python/
!pip install -q opencv-python

In [ ]:
# https://pypi.org/project/supervision/
!pip install -q supervision --upgrade

In [ ]:
!pip install ipywidgets --upgrade

In [ ]:
# Imports
import torch
import torchvision
import cv2
import supervision as sv
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor

In [ ]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "Data Science Academy" --iversions

### Download SAM Weights (Download do Modelo Pré-Treinado)

A célula abaixo precisa ser executada somente uma vez para download do modelo.

In [ ]:
#%cd {HOME}/modelo
#!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

In [ ]:
# Caminho onde está o modelo
caminho_modelo = os.path.join(HOME, "model", "sam_vit_h_4b8939.pth")

In [ ]:
# Caminho onde está o modelo
print(caminho_modelo, "; exist:", os.path.isfile(caminho_modelo))

## Carregando o Modelo

In [ ]:
# Define o device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [ ]:
print(f"Usando Device: {device}")

In [ ]:
# Se CUDA estiver instalado imprimimos detalhes da GPU
if torch.cuda.is_available():
    print('Número de CUDA Devices:', torch.cuda.device_count())
    print('Modelo da GPU:', torch.cuda.get_device_name(0))
    print('Total de Memória [GB] da GPU:', torch.cuda.get_device_properties(0).total_memory/1e9)

In [ ]:
# Tipo do modelo
tipo_modelo = "vit_h"

- vit_h = ViT Huge - https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
- vit_l = ViT Large - https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth
- vit_b = ViT Base - https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

In [ ]:
# Registra o modelo para uso nesta sessão do Jupyter Notebook
modelo_sam = sam_model_registry[tipo_modelo](checkpoint = caminho_modelo).to(device = device)

In [ ]:
print(modelo_sam)

## Geração Automatizada de Máscaras

Para executar a geração automática de máscaras, forneça um modelo SAM para a classe `SamAutomaticMaskGenerator`. Recomendado usar GPU.

In [ ]:
# Cria o gerador automático de máscara
mask_generator = SamAutomaticMaskGenerator(modelo_sam)

In [ ]:
# Imagens para segmentar
nome_imagem = "baleias.jpg"
#nome_imagem = "pessoas.jpg"
#nome_imagem = "frutas.png"
#nome_imagem = "carros1.jpg"
#nome_imagem = "carros2.jpg"
#nome_imagem = "carros3.jpg"
#nome_imagem = "brain1.jpg"
#nome_imagem = "brain2.png"
#nome_imagem = "lixo.png"
#nome_imagem = "fabrica1.jpg"
#nome_imagem = "fabrica2.jpeg"
#nome_imagem = "arvore.jpg"
#nome_imagem = "casa.jpg"

In [ ]:
# Caminho da imagem
caminho_imagem = os.path.join(HOME, "data", nome_imagem)

## Gerando Máscaras com SAM

In [ ]:
# Carrega a imagem
imagem_bgr = cv2.imread(caminho_imagem)

In [ ]:
# Converte o mapa de cores de BGR para RGB
imagem_rgb = cv2.cvtColor(imagem_bgr, cv2.COLOR_BGR2RGB)

In [ ]:
# Gera as máscaras
resultado = mask_generator.generate(imagem_rgb)

In [ ]:
print(resultado)

In [ ]:
# Vamos imprimir o resultado no índice zero
print(resultado[0].keys())

**Saída**:

SamAutomaticMaskGenerator() retorna uma lista de máscaras, onde cada máscara é um dicionário contendo várias informações:

* segmentation - `[np.ndarray]` - a máscara com shape `(W, H)` e tipo `bool` 
* area - `[int]` - a área da máscara em pixels
* bbox - `[List[int]]` - a caixa limite da máscara no formato `xywh`
* predicted_iou - `[float]` - a própria previsão do modelo para a qualidade da máscara
* point_coords - `[List[List[float]]]` - o ponto de entrada amostrado que gerou esta máscara
* stability_score - `[float]` - uma medida adicional da qualidade da máscara
* crop_box - `List[int]` - o recorte da imagem usada para gerar esta máscara no formato `xywh`

## Segmentando Objetos em Imagens de Forma Automática

Visualização de Resultados com Supervision

O Supervision possui suporte nativo para SAM.

In [ ]:
# Cria o objeto 
#mask_annotator = sv.MaskAnnotator()
mask_annotator = sv.MaskAnnotator(color_lookup = sv.ColorLookup.INDEX)

In [ ]:
# Extrai as máscaras detectadas com SAM
detections = sv.Detections.from_sam(sam_result = resultado)

In [ ]:
len(detections)

In [ ]:
# Busca as imagens anotadas. ATENÇÃO: Usamos aqui imagem_bgr
annotated_image = mask_annotator.annotate(scene = imagem_bgr.copy(), detections = detections)

In [ ]:
# Plot
sv.plot_images_grid(images = [imagem_bgr, annotated_image], 
                    grid_size = (1, 2), 
                    titles = ['Imagem Original', 'Imagem Segmentada Pelo SAM'])

## Visualizando as Máscaras Previstas Pelo SAM

In [ ]:
# Extrai as máscaras
masks = [mask['segmentation'] for mask in sorted(resultado, key = lambda x: x['area'], reverse = True)]

In [ ]:
# Plot das máscaras previstas pelo SAM
sv.plot_images_grid(images = masks, grid_size = (18, int(len(masks) / 8)), size = (16, 16))

# Fim